# controls

> FastHTML constructors for native buttons, inputs, groups, toolbars, copy actions, toggles, sliders, fields, and fieldsets styled from browser and ARIA states


In [ ]:
#| default_exp controls

These constructors return native HTML form controls marked with opt-in attributes such as `btn` or `input`. The browser manages control state. Shared CSS applies those states through pseudo-classes (`:disabled`, `:user-invalid`) and ARIA attributes (`aria-busy`, `aria-pressed`). Plain FastHTML `Button` and `Input` elements remain unstyled. The Python constructors emit markup only and include no stylesheets or browser scripts.

In [ ]:
from fastcore.utils import *
from fasthtml.common import Div, A, Label, Small, Form, Option, Legend, Code, unqid
from fasthtml.common import Button as _Button, Input as _Input, Textarea as _Textarea, Select as _Select, Fieldset as _Fieldset
from fastlucide.icons import Check, Copy as CopyIcon
from answerui.registry import hypelt
from answerui.demo import example
from answerui.layout import Stack, Cluster

In [ ]:
from fasthtml.common import *
from answerui.demo import demo
from answerui.registry import reg
from fastcore.test import *
from fastlucide.icons import X

## Btn

Buttons are raised at rest and sunken while pressed. Pressing shifts the label, bevels, and shadows without changing outer dimensions. A row of buttons stays aligned regardless of which ones are pressed.

Three ARIA attributes hold a button in its sunken state without shifting the label: `aria-busy="true"` for an in-flight request, `aria-pressed="true"` for a latched toggle, and `aria-expanded="true"` for an open popover or menu. Each variant controls that appearance through `--ui-held-face` and `--ui-held-shadow`.

A busy button fades its label and shows a centred spinner without resizing. Because `aria-busy="true"` changes appearance only, users can still click a busy button. Set `disabled` to prevent activation.

When a button triggers an htmx request, htmx adds the `htmx-request` class while the request runs. The busy appearance waits for `--busy-delay` (default `200ms`) to avoid flashing on fast requests. Server-set `aria-busy="true"` appears immediately. When a form triggers the request, htmx adds `htmx-request` to the form instead of the button. Set `hx-indicator` on the form to target the button.

A disabled button is flat and faded with no shadow while retaining its variant colour. Re-enabling it restores the raised appearance.

The default `primary` variant uses the accent colour. `variant='secondary'` uses the neutral surface colour with a hairline border. `variant='quiet'` has no background, border, or shadow at rest and inherits the parent text colour. Hovering a quiet button applies `--tint` (8% alpha). Pressing or holding it applies a 12% tint.

Pass `size='sm'` or `size='lg'` to set `btn-size` for compact or prominent actions (`None` uses the default size).

Pass `href` to render an `<a>` styled as a button, with the same variants, sizes, and `icon` option (omitting `type`). Because links have no native `disabled` state, `:disabled` styling does not apply to them. Handwritten `<a btn href="...">` elements match the same rules.

In [ ]:
@hypelt(sel='[btn]')
def Btn(
    *c,                # Button content
    variant='primary', # primary, secondary, quiet or editable
    size=None,         # sm or lg; None uses the default size
    href=None,         # Link URL; renders an <a> styled as a button
    type='button',     # Explicitly use submit for a form action (ignored with href)
    icon=False,        # Square button holding only an icon; supply an accessible name
    **attrs,           # Native button or anchor attributes
):
    "A raised action, latched by aria-pressed and sunken while busy or expanded"
    kw = dict(btn=True, btn_variant=variant, btn_size=size, btn_icon=icon, **attrs)
    return A(*c, href=href, **kw) if href is not None else _Button(*c, type=type, **kw)

Each variant, size, and state in a wrapping cluster: default, secondary, quiet, a link, small and large sizes, danger and success tones, disabled, busy, and latched:


In [ ]:
@example(Btn)
def _button_example():
    return Cluster(Btn('Save'), Btn('Cancel', variant='secondary'), Btn('Dismiss', variant='quiet'), Btn('Sign up', href='#'),
        Btn('Small', size='sm'), Btn('Large', size='lg'), Btn('Delete', ui_tone='danger'), Btn('Publish', ui_tone='success'),
        Btn('Locked', disabled=True), Btn('Save', aria_busy='true'), Btn('Pinned', aria_pressed='true'),
        Btn('Pinned', variant='secondary', aria_pressed='true'), Btn('Pinned', variant='quiet', aria_pressed='true'),
        cls='gap-2')


In [ ]:
Btn.demo()

```html
<div cluster style="--gap: var(--space-2);">
<button type="button" btn btn-variant="primary">Save</button><button type="button" btn btn-variant="secondary">Cancel</button><button type="button" btn btn-variant="quiet">Dismiss</button><a href="#" btn>Sign up</a><button type="button" btn btn-variant="primary" ui-tone="danger">Delete</button><button type="button" btn btn-variant="primary" ui-tone="success">Publish</button><button type="button" btn btn-variant="primary" disabled>Locked</button><button type="button" btn btn-variant="primary" aria-busy="true">Save</button><button type="button" btn btn-variant="primary" aria-pressed="true">Pinned</button><button type="button" btn btn-variant="secondary" aria-pressed="true">Pinned</button><button type="button" btn btn-variant="quiet" aria-pressed="true">Pinned</button></div>

```

Passing `href` emits an `<a>` without `type`, and `size` sets `btn-size` only when given:

In [ ]:
link = Btn('Sign up', href='/signup', variant='secondary', size='lg')
test_eq(link.tag, 'a')
assert 'type' not in link.attrs
test_eq(link.attrs['btn-size'], 'lg')
assert Btn('Save').attrs.get('btn-size') is None
to_xml(link)

Custom variants can override resting padding (`--ui-pad-block`, `--ui-pad-inline`) or `--relief` without redefining the pressed state. Below, `compact` tightens padding and `flat` sets `--relief: 0`. Both variants are registered on `Btn` for this notebook rather than built in. `doc(Btn)` lists them while they remain registered. To reduce depth across an entire application, set `:root { --relief: .5; }`.

In [ ]:
Btn.variant('compact', '--ui-pad-block: var(--space-3); --ui-pad-inline: var(--space-2);', doc='Tighter padding.')
Btn.variant('flat', '--relief: 0;', doc='No rims, shadows, or displacement.')

Btn.flat [app] :is([btn])[flat]: No rims, shadows, or displacement.

Flat buttons show their colour states without bevels, shadows, or label shift. The two Pin buttons compare unselected (`aria-pressed="false"`) and selected (`aria-pressed="true"`):

In [ ]:
demo(Cluster(
    Btn('Compact', compact=True), Btn('Flat', flat=True),
    Btn('Pin', variant='secondary', flat=True, aria_pressed='false'),
    Btn('Pin', variant='secondary', flat=True, aria_pressed='true'),
    cls='gap-2'))

```html
<div cluster style="--gap: var(--space-2);">
<button type="button" btn btn-variant="primary" compact>Compact</button><button type="button" btn btn-variant="primary" flat>Flat</button><button type="button" btn btn-variant="secondary" flat aria-pressed="false">Pin</button><button type="button" btn btn-variant="secondary" flat aria-pressed="true">Pin</button></div>

```

Pass `icon=True` for an icon-only button to make its horizontal padding match its vertical padding (`--ui-pad-inline: var(--ui-pad-block)`). Always supply an `aria-label` because an icon has no accessible name. Buttons containing both an icon and text use standard button padding.

Three square icon-only close buttons (`primary`, `secondary`, `quiet`) beside a labelled icon button and a text-only button:

In [ ]:
close = [Btn(X(), icon=True, aria_label='Close'), Btn(X(), icon=True, aria_label='Close', variant='secondary'),
    Btn(X(), icon=True, aria_label='Close', variant='quiet')]
demo(Cluster(*close, Btn(X(), 'Close', variant='secondary'), Btn('Save'), cls='gap-2'))

```html
<div cluster style="--gap: var(--space-2);">
<button type="button" btn btn-variant="primary" btn-icon aria-label="Close"><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg></button><button type="button" btn btn-variant="secondary" btn-icon aria-label="Close"><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg></button><button type="button" btn btn-variant="quiet" btn-icon aria-label="Close"><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg></button><button type="button" btn btn-variant="secondary"><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg>Close</button><button type="button" btn btn-variant="primary">Save</button></div>

```

## Text controls

Text controls use inverted bevel lighting (`--bevel-in` and `--sunken`) to look recessed into the page like a well. Disabled controls flatten to a muted background without the sunken shadow. `Input`, `Textarea`, and `Select` add the `input`, `textarea`, and `select` marker attributes. Handwritten HTML with the same attributes matches the same CSS rules. `Input` rejects `type='checkbox'`, `type='radio'`, and `type='range'` so well styling is not applied by mistake; use `Checkbox`, `Radio`, or `Range` instead, and use FastHTML's unmarked `Input` for file pickers or button-type inputs.

Number inputs (`type="number"`) hide browser spin buttons while keeping keyboard arrow stepping.

Invalid controls show a danger border colour computed from `--hue-danger`. That rule matches both `:user-invalid` (set by the browser after interaction) and server-set `aria-invalid="true"`:

In [ ]:
_input_alts = dict(checkbox='Checkbox', radio='Radio', range='Range')

@hypelt(sel='input[input]')
def Input(**attrs):
    "A text-like input styled as a well"
    if (t := attrs.get('type')) in _input_alts: raise ValueError(f"Use {_input_alts[t]}(...) instead of Input(type={t!r})")
    return _Input(input=True, **attrs)

@hypelt(sel='textarea[textarea]')
def Textarea(*c, **attrs):
    "A text area that grows with its content"
    return _Textarea(*c, textarea=True, **attrs)

@hypelt(sel='select[select]')
def Select(*c, **attrs):
    "A native select styled as a well"
    return _Select(*c, select=True, **attrs)

Each text control on its own, at rest, and disabled:

In [ ]:
@example(Input)
def _input_example(): return Stack(Input(placeholder='Search', type='search', aria_label='Search'), Input(value='Locked', disabled=True, aria_label='Locked'), style='inline-size:16rem')

@example(Textarea)
def _textarea_example(): return Textarea('A note that grows with its content.', aria_label='Note', style='inline-size:16rem')

@example(Select)
def _select_example(): return Select(Option('Free'), Option('Team'), aria_label='Plan', style='inline-size:16rem')

In [ ]:
Input.demo()

```html
<div stack style="inline-size:16rem">
  <input input placeholder="Search" type="search" aria-label="Search">
  <input input value="Locked" disabled aria-label="Locked">
</div>

```

In [ ]:
for t, alt in _input_alts.items(): test_fail(lambda t=t: Input(type=t), exc=ValueError, contains=alt)

Add the `input` attribute to style a non-control element as a well, such as a read-only block or a `div` hosting a third-party widget:

In [ ]:
demo(Stack(Div('A widget mounts here', input=True), Input(placeholder='A text input, for comparison', aria_label='Comparison')))

```html
<div stack>
  <div input>A widget mounts here</div>
  <input input placeholder="A text input, for comparison" aria-label="Comparison">
</div>

```

## Enabling a button

Render a submit button `disabled` with an `enable` attribute to enable it automatically when form conditions are met. Set `enable="changed"` to wait until any control differs from its initial rendered value. Set `enable="valid"` to wait until the form passes native validation, or `enable="changed valid"` to require both.

The shared form handler compares each control against its browser default (`defaultValue`, `defaultChecked`, or `defaultSelected`) without storing a separate snapshot. Resetting the form disables the button again. Forms swapped in by htmx work the same way because the browser treats their rendered values as defaults.


Change the name or enter a valid required email address to enable Save. Clicking Reset restores the initial values and disables Save again:


In [ ]:
demo(Form(Input(name='name', value='Ada', aria_label='Display name'), Input(name='email', type='email', required=True, placeholder='Email', aria_label='Email'),
    Cluster(Btn('Save', type='submit', disabled=True, enable='changed valid'), Btn('Reset', type='reset', variant='secondary')), stack=True, style='inline-size: 24rem'))


```html
<form enctype="multipart/form-data" stack style="inline-size: 24rem">  <input input name="name" value="Ada" aria-label="Display name">
  <input input name="email" type="email" required placeholder="Email" aria-label="Email">
  <div cluster>
<button type="submit" btn btn-variant="primary" disabled enable="changed valid">Save</button><button type="reset" btn btn-variant="secondary">Reset</button>  </div>
</form>
```

## Group

`Group` wraps related controls in a `div[group]` with shared inner borders and rounded outer corners. Passing `inset=True` sets `group-variant="inset"` to render the whole group as a single well with the action button inside.

`Toolbar` wraps controls in a `div[toolbar][group]` with `role="toolbar"`. The shared toolbar controller provides a single Tab stop and moves focus across enabled commands with Left, Right, Home, and End. Set `aria-pressed` on the server to mark latched toggles. Use native radios for an exclusive form choice.


In [ ]:
@hypelt(sel='[group]')
def Group(*c, inset=False, **attrs):
    "Controls sharing an edge; inset places actions inside the input's well"
    return Div(*c, group=True, group_variant='inset' if inset else None, **attrs)

@hypelt(sel='[toolbar]')
def Toolbar(*controls, **attrs):
    "A row of commands with arrow-key navigation"
    return Div(*controls, toolbar=True, role='toolbar', group=True, **attrs)

A standard group (name input with Create button) and an inset group (editable name with Save button). Enter a name in the first to enable Create, or edit the second to enable Save:


In [ ]:
@example(Group)
def _group_example():
    return Stack(Form(Group(Input(name='name', placeholder='Instance name', required=True, aria_label='Instance name'), Btn('Create', type='submit', disabled=True, enable='valid'))),
        Form(Group(Input(name='name', value='personal', aria_label='Name'), Btn('Save', type='submit', disabled=True, enable='changed valid'), inset=True)),
        style='inline-size: 24rem')

In [ ]:
Group.demo()

```html
<div stack style="inline-size: 24rem">
<form enctype="multipart/form-data">    <div group>
      <input input name="name" placeholder="Instance name" required aria-label="Instance name">
<button type="submit" btn btn-variant="primary" disabled enable="valid">Create</button>    </div>
</form><form enctype="multipart/form-data">    <div group group-variant="inset">
      <input input name="name" value="personal" aria-label="Name">
<button type="submit" btn btn-variant="primary" disabled enable="changed valid">Save</button>    </div>
</form></div>

```

A pager toolbar and a toggle bar labelled with `aria-label`, with one button latched and one disabled. In production the server sets `aria-pressed`. Here a demo-only rule toggles it on click:


In [ ]:
reg.rule(name='demo-toolbar-toggle', sel='[toolbar]', on={('& button[aria-pressed]', 'click'): r"el.setAttribute('aria-pressed', String(el.getAttribute('aria-pressed') !== 'true'))"},
    doc='Demo only: flips aria-pressed in place of the server.')

demo-toolbar-toggle [app] [toolbar]: Demo only: flips aria-pressed in place of the server.

In [ ]:
@example(Toolbar)
def _toolbar_example():
    return Cluster(Toolbar(Btn('Previous', variant='secondary'), Btn('Next', variant='secondary'), aria_label='Pages'),
        Toolbar(Btn('Bold', variant='secondary', aria_pressed='true'), Btn('Italic', variant='secondary', aria_pressed='false'),
            Btn('Underline', variant='secondary', aria_pressed='false', disabled=True), aria_label='Text style'))

In [ ]:
Toolbar.demo()

```html
<div cluster>
  <div toolbar role="toolbar" group aria-label="Pages">
<button type="button" btn btn-variant="secondary">Previous</button><button type="button" btn btn-variant="secondary">Next</button>  </div>
  <div toolbar role="toolbar" group aria-label="Text style">
<button type="button" btn btn-variant="secondary" aria-pressed="true">Bold</button><button type="button" btn btn-variant="secondary" aria-pressed="false">Italic</button><button type="button" btn btn-variant="secondary" aria-pressed="false" disabled>Underline</button>  </div>
</div>

```

## Copy

`Copy` renders a `quiet` button (icon-only by default, with `aria-label="Copy"`) that copies text to the clipboard. Pass `copyfor` with the `id` of a target element to copy a form control's `value` or any other element's `textContent`. Pass `copy` to copy a fixed string instead.

For `1.5s` after clicking, the click handler sets the `copied` attribute on the button and swaps its icon to a check mark. If the button has an `aria-label`, the handler temporarily sets it to `'Copied'` so screen readers announce the confirmation. Buttons with visible text labels are left unchanged. The Clipboard API requires a secure context (HTTPS or localhost).

In [ ]:
@hypelt(sel='button:is([copyfor], [copy])')
def Copy(
    *c,              # Label text; with none, a square button named Copy
    copyfor=None,    # Id of the element whose value or text is copied
    copy=None,       # Fixed text to copy instead of an element's
    variant='quiet', # Button appearance
    **attrs,         # Native button attributes
):
    "Copy fixed text or an element's value or text to the clipboard"
    if not c: attrs.setdefault('aria_label', 'Copy')
    return Btn(CopyIcon(aria_hidden='true') + Check(), *c, copyfor=copyfor, copy=copy, variant=variant, icon=not c, **attrs)

An API key inside a read-only inset group with an icon-only copy button, followed by a command beside a labelled `secondary` copy button. Click either button to test the check state:

In [ ]:
@example(Copy)
def _copy_example():
    key, cmd = unqid(), unqid()
    return Stack(Group(Input(value='sk-live-4f9c2e7b1a0d', readonly=True, id=key, aria_label='API key'), Copy(copyfor=key), inset=True),
        Cluster(Code('pip install answerui-py', id=cmd), Copy('Copy', copyfor=cmd, variant='secondary'), cls='gap-2'),
        style='inline-size: 24rem')

In [ ]:
Copy.demo()

```html
<div stack style="inline-size: 24rem">
  <div group group-variant="inset">
    <input input value="sk-live-4f9c2e7b1a0d" readonly aria-label="API key" id="_2XOIYcptSza5zxci_WP8zQ" name="_2XOIYcptSza5zxci_WP8zQ">
<button type="button" btn btn-variant="quiet" btn-icon copyfor="_2XOIYcptSza5zxci_WP8zQ" aria-label="Copy"><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" aria-hidden="true" class="lucide-icon "><use href="#lc-copy"></use><use href="#lc-check"></use></svg></button>  </div>
  <div cluster style="--gap: var(--space-2);">
<code id="_HSbHk9uoS5GDNDEc7OHeAA">pip install answerui-py</code><button type="button" btn btn-variant="secondary" copyfor="_HSbHk9uoS5GDNDEc7OHeAA"><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" aria-hidden="true" class="lucide-icon "><use href="#lc-copy"></use><use href="#lc-check"></use></svg>Copy</button>  </div>
</div>

```

## Click to edit

Use `Btn(..., variant='editable')` for values that are read far more often than edited. It renders a full-width button with the same padding and corner radius as a text well, bordered by a dashed hairline (`--n`) instead of a sunken surface. No input exists until the user clicks. Visibility does not depend on hover. On click, the application fetches and swaps in either a single input or a multi-field form. Inside a form, keep `type="button"` (the default on `Btn`). Set `aria-label` to describe the edit action because the button's visible text is the current value.


The demo below uses a client-only rule to swap each value with its edit form. Click the name to open an inset input with Save, or click the schedule to edit day and time. Submitting updates the displayed value. Pressing Escape or clicking outside cancels and restores the previous value:


In [ ]:
reg.rule(name='demo-editable', sel='[editable-demo]', on={
    ('& button[btn-variant=editable]', 'click'): r"const p = el.closest('[editable-demo]'); p.firstElementChild.hidden = true; p.lastElementChild.hidden = false; p.querySelector('input, select').focus()",
    ('& form', 'submit'): r"e.preventDefault(); const p = el.closest('[editable-demo]'); p.querySelector('button[btn-variant=editable]').textContent = el.dataset.text.replace(/\{(\w+)\}/g, (_, n) => el.elements[n].value); el.hidden = true; p.firstElementChild.hidden = false",
    ('& form', 'keydown'): r"if (e.key === 'Escape') { el.reset(); el.hidden = true; el.previousElementSibling.hidden = false }",
    (':root', 'pointerdown'): r"for (const f of document.querySelectorAll('[editable-demo] > form:not([hidden])')) if (!f.contains(e.target)) { f.reset(); f.hidden = true; f.previousElementSibling.hidden = false }",
}, doc='Demo only: swaps the value for its form and back in place of the server; a click outside or Escape cancels.')

def _editable_demo(value, form, text, **attrs):
    "An editable value and the hidden form that edits it; `text` is the template that turns the form's fields into the new value"
    return Div(Btn(value, variant='editable', **attrs), Form(form, data_text=text, hidden=True), editable_demo=True)


In [ ]:
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
demo(Stack(
    _editable_demo('Untitled-1', Group(Input(name='name', value='Untitled-1', aria_label='Name'), Btn('Save', type='submit', disabled=True, enable='changed valid'), inset=True),
        '{name}', aria_label='Rename Untitled-1'),
    _editable_demo('Every Monday at 09:00', Group(Select(*[Option(d, selected=d == 'Monday') for d in days], name='day', aria_label='Day'),
        Input(type='time', name='time', value='09:00', aria_label='Time'), Btn('Save', type='submit', disabled=True, enable='changed valid')),
        'Every {day} at {time}', aria_label='Change the schedule'),
    style='inline-size: 24rem'))


```html
<div stack style="inline-size: 24rem">
  <div editable-demo>
<button type="button" btn btn-variant="editable" aria-label="Rename Untitled-1">Untitled-1</button><form enctype="multipart/form-data" data-text="{name}" hidden>      <div group group-variant="inset">
        <input input name="name" value="Untitled-1" aria-label="Name">
<button type="submit" btn btn-variant="primary" disabled enable="changed valid">Save</button>      </div>
</form>  </div>
  <div editable-demo>
<button type="button" btn btn-variant="editable" aria-label="Change the schedule">Every Monday at 09:00</button><form enctype="multipart/form-data" data-text="Every {day} at {time}" hidden>      <div group>
<select select name="day" aria-label="Day"><option selected>Monday</option><option>Tuesday</option><option>Wednesday</option><option>Thursday</option><option>Friday</option></select>        <input input type="time" name="time" value="09:00" aria-label="Time">
<button type="submit" btn btn-variant="primary" disabled enable="changed valid">Save</button>      </div>
</form>  </div>
</div>

```

## Switch

Use `Switch` for immediate on/off settings rather than deferred form submissions. `Switch` returns a native `<input type="checkbox" role="switch" switch>`, letting the browser handle state, keyboard input, and accessibility announcements. CSS styles the track as a recessed well with a raised thumb that slides across on `--color-accent` when checked. Inside `Field`, the control appears before its label just like a checkbox:

In [ ]:
@hypelt(sel='input[switch]')
def Switch(**attrs):
    "A checkbox with the switch role"
    return _Input(type='checkbox', role='switch', switch=True, **attrs)

Off, on, and disabled:

In [ ]:
@example(Switch)
def _switch_example():
    return Cluster(Switch(aria_label='Off'), Switch(checked=True, aria_label='On'), Switch(disabled=True, aria_label='Disabled'))

In [ ]:
Switch.demo()

```html
<div cluster>
  <input type="checkbox" role="switch" switch aria-label="Off">
  <input type="checkbox" role="switch" switch checked aria-label="On">
  <input type="checkbox" role="switch" switch disabled aria-label="Disabled">
</div>

```

## Checkbox and radio

`Checkbox` and `Radio` return native `<input type="checkbox" checkbox>` and `<input type="radio" radio>` elements. Shared CSS sets `appearance: none` to style each as a small recessed well with `--sunken` shadow and no bevel rims (reverting to native `appearance: auto` under `forced-colors: active`):

In [ ]:
@hypelt(sel='input[checkbox]')
def Checkbox(**attrs):
    "A checkbox styled as a small well"
    return _Input(type='checkbox', checkbox=True, **attrs)


In [ ]:
@hypelt(sel='input[radio]')
def Radio(**attrs):
    "A radio button styled as a small well"
    return _Input(type='radio', radio=True, **attrs)

Unchecked, checked, disabled, and checked and disabled:

In [ ]:
@example(Checkbox)
def _checkbox_example():
    return Cluster(Checkbox(aria_label='Unchecked'), Checkbox(checked=True, aria_label='Checked'),
        Checkbox(disabled=True, aria_label='Disabled'), Checkbox(checked=True, disabled=True, aria_label='Checked and disabled'))


In [ ]:
Checkbox.demo()

```html
<div cluster>
  <input type="checkbox" checkbox aria-label="Unchecked">
  <input type="checkbox" checkbox checked aria-label="Checked">
  <input type="checkbox" checkbox disabled aria-label="Disabled">
  <input type="checkbox" checkbox checked disabled aria-label="Checked and disabled">
</div>

```

Radios in a group, unchecked, checked, and checked and disabled:

In [ ]:
@example(Radio)
def _radio_example():
    return Cluster(Radio(name='r', aria_label='Unchecked'), Radio(name='r', checked=True, aria_label='Checked'), Radio(name='rd', checked=True, disabled=True, aria_label='Checked and disabled'))

In [ ]:
Radio.demo()

```html
<div cluster>
  <input type="radio" radio name="r" aria-label="Unchecked">
  <input type="radio" radio name="r" checked aria-label="Checked">
  <input type="radio" radio name="rd" checked disabled aria-label="Checked and disabled">
</div>

```

## Range

`Range` returns an `<input type="range" range>` styled as a recessed track with a raised thumb. The shared browser script computes `--ui-range-fill` from `value`, `min` (default `0`), and `max` (default `100`) on mount, on `input` or `change`, and after form reset. Scripts that assign `value` programmatically must dispatch an `input` event to refresh the track fill.

The shared stylesheet (`03_controls`) defines the WebKit and Firefox track and thumb rules once. `Range` adds only the `range` attribute.

In [ ]:
@hypelt(sel='input[range]')
def Range(**attrs):
    "A slider with a filled track and raised thumb"
    return _Input(type='range', range=True, **attrs)

Four sliders below show default (`35`), disabled (`65`), `0` to `10` (`8`), and `-10` to `0` (`-5`). The last slider confirms that `max=0` acts as an explicit upper bound rather than falling back to `100`:

In [ ]:
@example(Range)
def _range_example():
    return Stack(Range(value=35, aria_label='Brightness'), Range(value=65, disabled=True, aria_label='Locked'), Range(min=0, max=10, value=8, aria_label='Rating'),
        Range(min=-10, max=0, value=-5, aria_label='Offset'),
        style='inline-size:16rem')

In [ ]:
Range.demo()

```html
<div stack style="inline-size:16rem">
  <input type="range" range value="35" aria-label="Brightness">
  <input type="range" range value="65" disabled aria-label="Locked">
  <input type="range" range min="0" max="10" value="8" aria-label="Rating">
  <input type="range" range min="-10" max="0" value="-5" aria-label="Offset">
</div>

```

## Field

`Field` pairs a control with its `<label>` plus optional `hint` and `error` messages so assistive technology announces them together. If the control lacks an `id`, `Field` generates one and links the `<label>` via `for`. `Field` sets `field-inline` on switches, checkboxes, and radios so the control appears before its label text. Because that inline label uses `justify-self: start`, clicking empty space to the right of the text does not toggle the control.

`hint` and `error` render as `<small>` elements below the control, with their ids appended to the control's `aria-describedby`. Passing `error` styles the message with `ui-tone="danger"` and sets `aria-invalid="true"` on the control. Validation rules stay in the application. `Field` only displays the message you pass.


In [ ]:
@hypelt(sel='[field]')
def Field(
    label,      # Label text
    control,    # The control
    hint=None,  # Help text shown under the control
    error=None, # Error message; marks the control invalid
    **attrs,    # Attributes for the field
):
    "A labelled control with optional help and error text"
    if not control.get('id'): control(id=unqid())
    descriptions = []
    if hint: descriptions.append(Small(hint, id=True))
    if error:
        descriptions.append(Small(error, id=True, ui_tone='danger'))
        control(aria_invalid='true')
    if descriptions:
        ids = [control.get('aria-describedby', ''), *(o.id for o in descriptions)]
        control(aria_describedby=' '.join(filter(None, ids)))
    return Div(Label(label, fr=control.id), Div(control, field_control=True), *descriptions,
        field=True, field_inline=control.get('type') in ('checkbox', 'radio'), **attrs)


Native browser validation bubbles appear only on submit and never show when `enable="valid"` disables the submit button. Instead, the shared field handler renders the browser's `validationMessage` inside `<small validation ui-tone="danger">` under `[field]` when the user leaves a `:user-invalid` control or when submission fails. On a failed submit, every invalid field displays its message and the handler focuses the first invalid control.

While displayed, the validation element's `id` is added to the control's `aria-describedby`. Once the value becomes valid, the handler removes the `<small validation>` element and strips its `id` from `aria-describedby`. Server-rendered `error` messages passed to `Field` have no `validation` attribute and are left untouched by the client handler.

For form-level server errors, render `<small validation ui-tone="danger">` (`Small(msg, ui_tone='danger', validation=True)`) inside the form outside any `[field]`. Editing any control in the form removes that form-level message immediately. A `<small validation>` element inside a `[field]` is cleared only when that field's control becomes valid.

All control types in one form. The Username field shows both a `hint` and a server-supplied `error`. Enter an invalid address in Email and move focus away to trigger client validation:

In [ ]:
@example(Field)
def _field_example():
    return Form(
        Field('Name', Input(name='name', placeholder='Ada Lovelace')),
        Field('Email', Input(name='email', type='email', required=True), hint='We never share it.'),
        Field('Notes', Textarea(name='notes', placeholder='Anything else?')),
        Field('Plan', Select(Option('Free'), Option('Team'), name='plan')),
        Field('Send me updates', Checkbox(name='updates', checked=True)),
        Field('Email me about mentions', Switch(name='mentions', checked=True)),
        Field('Weekly digest', Radio(name='digest', checked=True)),
        Field('Volume', Range(name='volume')),
        Field('Region', Input(name='region', value='Locked', disabled=True)),
        Field('Username', Input(name='username', value='taken'), hint='Shown on your profile.', error='That name is in use.'),
        Cluster(Btn('Save', type='submit'), Btn('Cancel', variant='secondary'), cls='gap-2'),
        stack=True)


In [ ]:
Field.demo()

```html
<form enctype="multipart/form-data" stack>  <div field>
<label for="_I2hgXzuSRMWbBFGb5I-Ecw">Name</label>    <div field-control>
      <input input name="name" placeholder="Ada Lovelace" id="_I2hgXzuSRMWbBFGb5I-Ecw">
    </div>
  </div>
  <div field>
<label for="_w0263pMDTjuQRU_de7ATZA">Email</label>    <div field-control>
      <input input name="email" type="email" required id="_w0263pMDTjuQRU_de7ATZA" aria-describedby="_i185QoJIT1Ctd23desl5xw">
    </div>
<small id="_i185QoJIT1Ctd23desl5xw">We never share it.</small>  </div>
  <div field>
<label for="_zbAZtlwfTy2MCs5Z0qmJsQ">Notes</label>    <div field-control>
<textarea textarea name="notes" placeholder="Anything else?" id="_zbAZtlwfTy2MCs5Z0qmJsQ"></textarea>    </div>
  </div>
  <div field>
<label for="_kTqQQER1QYW-BN0ZNEjuTw">Plan</label>    <div field-control>
<select select name="plan" id="_kTqQQER1QYW-BN0ZNEjuTw"><option>Free</option><option>Team</option></select>    </div>
  </div>
  <div field field-inline>
<label for="_Z_aEUHR-SOGvMJiXffcZCw">Send me updates</label>    <div field-control>
      <input type="checkbox" checkbox name="updates" checked id="_Z_aEUHR-SOGvMJiXffcZCw">
    </div>
  </div>
  <div field field-inline>
<label for="_D3yWRfs6SxecIYVOCBfmow">Email me about mentions</label>    <div field-control>
      <input type="checkbox" role="switch" switch name="mentions" checked id="_D3yWRfs6SxecIYVOCBfmow">
    </div>
  </div>
  <div field field-inline>
<label for="_cDwPUpg3QbK7LIaAiLuvnw">Weekly digest</label>    <div field-control>
      <input type="radio" radio name="digest" checked id="_cDwPUpg3QbK7LIaAiLuvnw">
    </div>
  </div>
  <div field>
<label for="_lJnMmbmQREmDt2TUZHFWug">Volume</label>    <div field-control>
      <input type="range" range name="volume" id="_lJnMmbmQREmDt2TUZHFWug">
    </div>
  </div>
  <div field>
<label for="_sql8KIh2Q3K_URGzNQ4FiQ">Region</label>    <div field-control>
      <input input name="region" value="Locked" disabled id="_sql8KIh2Q3K_URGzNQ4FiQ">
    </div>
  </div>
  <div field>
<label for="_tVy9hB_UQ52W1lc5QoEYCQ">Username</label>    <div field-control>
      <input input name="username" value="taken" id="_tVy9hB_UQ52W1lc5QoEYCQ" aria-invalid="true" aria-describedby="_WdlNyJwpQ-WY-kT0haiXVA _jbmDMdekTGSwIX63hy_62A">
    </div>
<small id="_WdlNyJwpQ-WY-kT0haiXVA">Shown on your profile.</small><small ui-tone="danger" id="_jbmDMdekTGSwIX63hy_62A">That name is in use.</small>  </div>
  <div cluster style="--gap: var(--space-2);">
<button type="submit" btn btn-variant="primary">Save</button><button type="button" btn btn-variant="secondary">Cancel</button>  </div>
</form>
```

Extra keyword arguments (and variants declared on `Field`) are placed on the outer `div[field]`. Existing attributes on the control remain intact. Any existing `aria-describedby` ids precede the generated hint and error ids:

In [ ]:
notes = Textarea(name='notes', aria_describedby='notes-policy')
field = Field('Notes', notes, hint='Keep it brief.', error='Please remove the account number.', compact=True)
assert isinstance(notes.id, str)
test_eq(field.children[0].get('for'), notes.id)
assert notes.attrs['aria-invalid'] == 'true'
assert notes.attrs['aria-describedby'].split() == ['notes-policy', *(o.id for o in field.children[2:])]
test_eq(field.attrs['compact'], True)
notes.attrs


{'textarea': True,
 'name': 'notes',
 'aria-describedby': 'notes-policy __9f5p1xWReWmyC6Vs-bFwA _ucxBoihNQf_OucOyN2Oelw',
 'id': '_vGXJivcnSMioXsXipG9zpQ',
 'aria-invalid': 'true'}

## Fieldset

`Fieldset` renders a native `<fieldset fieldset>` with a hairline border and inner padding (`--space`) around a `<legend>`. Pass `stack=True` or another layout attribute to space the fields inside:

In [ ]:
@hypelt(sel='fieldset[fieldset]')
def Fieldset(*c, **attrs):
    "A bordered group of fields under a legend"
    return _Fieldset(*c, fieldset=True, **attrs)

Two checkbox fields under a legend:

In [ ]:
@example(Fieldset)
def _fieldset_example():
    return Fieldset(Legend('Notifications'),
        Field('Email me about mentions', Checkbox(checked=True)),
        Field('Email me a weekly digest', Checkbox()),
        stack=True, cls='gap-2')

In [ ]:
Fieldset.demo()

```html
<fieldset fieldset stack style="--gap: var(--space-2)"><legend>Notifications</legend>  <div field field-inline>
<label for="_p78SAPtxSMebn29wYLb3IQ">Email me about mentions</label>    <div field-control>
      <input type="checkbox" checkbox checked id="_p78SAPtxSMebn29wYLb3IQ">
    </div>
  </div>
  <div field field-inline>
<label for="_JL4GGjioQLGhocRzRFRRng">Email me a weekly digest</label>    <div field-control>
      <input type="checkbox" checkbox id="_JL4GGjioQLGhocRzRFRRng">
    </div>
  </div>
</fieldset>
```

To show a field only while a checkbox is checked, set `reveals` on the checkbox to the target element's `id`. Match the target's initial `hidden` attribute to the checkbox's initial checked state. On each `input` or `change` event, the shared handler sets `target.hidden = !checkbox.checked`:

In [ ]:
demo(Form(Field('Set a monthly limit', Checkbox(name='limit_on', reveals='limit')),
    Field('Monthly limit (US$)', Input(name='limit', type='number', value=200), id='limit', hidden=True), stack=True))

```html
<form enctype="multipart/form-data" stack>  <div field field-inline>
<label for="_FhAVw7fERk2Kj-6ek8X7GQ">Set a monthly limit</label>    <div field-control>
      <input type="checkbox" checkbox name="limit_on" reveals="limit" id="_FhAVw7fERk2Kj-6ek8X7GQ">
    </div>
  </div>
  <div field hidden id="limit">
<label for="_6ZC4nn66Q8aBB6zAhlFDUA">Monthly limit (US$)</label>    <div field-control>
      <input input name="limit" type="number" value="200" id="_6ZC4nn66Q8aBB6zAhlFDUA">
    </div>
  </div>
</form>
```